# 08 — Codegen Analysis


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from build_optimiser.config import Config
from build_optimiser.graph import load_graph, attach_metrics
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

cfg = Config.from_yaml('../config.yaml')
file_df = pd.read_parquet('../data/processed/file_metrics.parquet')
target_df = pd.read_parquet('../data/processed/target_metrics.parquet')
edge_df = pd.read_parquet('../data/processed/edge_list.parquet')
G = load_graph('../data/processed/edge_list.parquet')
attach_metrics(G, target_df)

%matplotlib inline
sns.set_theme(style='whitegrid')

## Codegen Inventory Summary

Total generated files, SLOC, compile time, % of total build cost.


In [ ]:
# --- Codegen Inventory Summary ---

def pct(part, whole):
    return f"{part / whole:.1%}" if whole > 0 else "N/A"

gen_files = file_df[file_df['is_generated']]
authored_files = file_df[~file_df['is_generated']]
codegen_targets = target_df[target_df['codegen_file_count'] > 0].copy()

# Scalars
total_files = len(file_df)
total_gen = len(gen_files)
total_authored = len(authored_files)

total_sloc = int(target_df['code_lines_total'].sum())
gen_sloc = int(target_df['code_lines_generated'].sum())

total_compile_ms = int(target_df['compile_time_sum_ms'].sum())
gen_compile_ms = int(target_df['codegen_compile_time_sum_ms'].sum())

total_build_ms = int(target_df['total_build_time_ms'].sum())
total_codegen_step_ms = int(target_df['codegen_time_ms'].fillna(0).sum())

print("=" * 70)
print(f"{'Metric':<35} {'Total':>10} {'Codegen':>10} {'%':>8}")
print("-" * 70)
print(f"{'Source files':<35} {total_files:>10,} {total_gen:>10,} {pct(total_gen, total_files):>8}")
print(f"{'SLOC':<35} {total_sloc:>10,} {gen_sloc:>10,} {pct(gen_sloc, total_sloc):>8}")
print(f"{'Compile time (ms)':<35} {total_compile_ms:>10,} {gen_compile_ms:>10,} {pct(gen_compile_ms, total_compile_ms):>8}")
print(f"{'Codegen step time (ms)':<35} {total_build_ms:>10,} {total_codegen_step_ms:>10,} {pct(total_codegen_step_ms, total_build_ms):>8}")
print(f"{'Targets with codegen':<35} {len(target_df):>10,} {len(codegen_targets):>10,} {pct(len(codegen_targets), len(target_df)):>8}")
print("=" * 70)

# --- Per-target breakdown ---
codegen_targets = codegen_targets.sort_values('codegen_compile_time_sum_ms', ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.barh(codegen_targets['cmake_target'], codegen_targets['codegen_file_count'], color='darkorange')
ax1.barh(codegen_targets['cmake_target'], codegen_targets['authored_file_count'],
         left=codegen_targets['codegen_file_count'], color='steelblue')
ax1.set_xlabel('File count')
ax1.set_title('Files per target (generated vs authored)')
ax1.legend(['Generated', 'Authored'], loc='lower right')
ax1.invert_yaxis()

ax2.barh(codegen_targets['cmake_target'], codegen_targets['codegen_compile_time_sum_ms'], color='darkorange')
ax2.barh(codegen_targets['cmake_target'], codegen_targets['authored_compile_time_sum_ms'],
         left=codegen_targets['codegen_compile_time_sum_ms'], color='steelblue')
ax2.set_xlabel('Compile time (ms)')
ax2.set_title('Compile time per target (generated vs authored)')
ax2.legend(['Generated', 'Authored'], loc='lower right')
ax2.invert_yaxis()

fig.suptitle('Codegen Inventory — Per-Target Breakdown', fontsize=14)
plt.tight_layout()
plt.show()

# --- Donut charts ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].pie([total_gen, total_authored], labels=['Generated', 'Authored'],
            colors=['darkorange', 'steelblue'], autopct='%1.1f%%', startangle=90,
            wedgeprops={'width': 0.5})
axes[0].set_title('Files')

axes[1].pie([gen_sloc, total_sloc - gen_sloc], labels=['Generated', 'Authored'],
            colors=['darkorange', 'steelblue'], autopct='%1.1f%%', startangle=90,
            wedgeprops={'width': 0.5})
axes[1].set_title('SLOC')

axes[2].pie([gen_compile_ms, total_compile_ms - gen_compile_ms], labels=['Generated', 'Authored'],
            colors=['darkorange', 'steelblue'], autopct='%1.1f%%', startangle=90,
            wedgeprops={'width': 0.5})
axes[2].set_title('Compile Time')

fig.suptitle('Codegen Share of Build', fontsize=14)
plt.tight_layout()
plt.show()

## Codegen Timing Analysis

Ninja log codegen step durations. Which generators are slowest?


In [ ]:
# TODO: Codegen Timing Analysis


## Codegen Fan-Out Analysis

How many targets consume each codegen output?


In [ ]:
# TODO: Codegen Fan-Out Analysis


## Generated File Compilation Cost

Compare compile time, preprocessed size, header depth: generated vs authored.


In [ ]:
# TODO: Generated File Compilation Cost


## Codegen-Triggered Rebuild Analysis

Total rebuild cost per codegen step x change frequency.


In [ ]:
# TODO: Codegen-Triggered Rebuild Analysis


## Codegen Dependency Analysis

Transitive codegen exposure — how many targets affected by codegen change?


In [ ]:
# TODO: Codegen Dependency Analysis


## Recommendations

Caching, parallelism, generated file optimisation, consolidation, isolation.


In [ ]:
# TODO: Recommendations
